In [1]:
import pandas as pd

load_df = pd.read_excel('/Users/shreya/Documents/College/Sem 5/ML/Lab/GridSense/data/raw/hourlyLoadDataIndia.xlsx')
temp_df = pd.read_excel('/Users/shreya/Documents/College/Sem 5/ML/Lab/GridSense/data/raw/monthly_temp.xlsx')

load_df.head()
load_df.info()
load_df.describe()
print(load_df['datetime'].min(), load_df['datetime'].max())  # adjust column name

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46728 entries, 0 to 46727
Data columns (total 7 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   datetime                            46728 non-null  datetime64[ns]
 1   National Hourly Demand              46728 non-null  float64       
 2   Northen Region Hourly Demand        46728 non-null  float64       
 3   Western Region Hourly Demand        46728 non-null  float64       
 4   Eastern Region Hourly Demand        46728 non-null  float64       
 5   Southern Region Hourly Demand       46728 non-null  float64       
 6   North-Eastern Region Hourly Demand  46728 non-null  float64       
dtypes: datetime64[ns](1), float64(6)
memory usage: 2.5 MB
2019-01-01 00:00:00 2024-04-30 23:00:00


In [2]:
# Check for missing timestamps (gaps)
full_range = pd.date_range(start=load_df['datetime'].min(), 
                             end=load_df['datetime'].max(), freq='h')
missing = full_range.difference(load_df['datetime'])
print(f"Missing hours: {len(missing)}")
print(missing[:20])  # peek at a few if any exist

# Check for duplicate timestamps
print(f"Duplicates: {load_df['datetime'].duplicated().sum()}")

# Sanity check the demand values themselves
load_df.describe()
# look for: negative demand (impossible), suspicious zeros, wild outliers

Missing hours: 0
DatetimeIndex([], dtype='datetime64[ns]', freq='h')
Duplicates: 0


,datetime,National Hourly Demand,Northen Region Hourly Demand,Western Region Hourly Demand,Eastern Region Hourly Demand,Southern Region Hourly Demand,North-Eastern Region Hourly Demand
count,46728,46728.000000,46728.000000,46728.000000,46728.000000,46728.000000,46728.00000
mean,2021-08-31 11:29:59.999999744,160487.065667,47914.796244,50402.921233,18572.519716,41589.467904,2007.36047
min,2019-01-01 00:00:00,95336.560000,19217.760000,28111.220000,9259.820000,23188.920000,567.00000
25%,2020-05-01 17:45:00,145338.782500,40165.942500,44272.367500,16107.155000,36156.870000,1689.17750
50%,2021-08-31 11:30:00,159541.060000,47523.690000,50036.970000,18427.610000,40681.885000,1968.65000
75%,2022-12-31 05:15:00,176633.607500,55012.182500,56100.490000,20828.360000,46101.767500,2290.20250
max,2024-04-30 23:00:00,237361.970000,80793.890000,73015.180000,28856.630000,68207.940000,3586.43000
std,NaN,23103.574609,10502.892378,7995.619137,3329.680794,7466.994359,436.58010


In [3]:
import pandas as pd
import pyarrow as pa

print(pd.__version__)
print(pa.__version__)

2.3.3
25.0.1


In [5]:
load_long = load_df.melt(
    id_vars=['datetime'],
    value_vars=['National Hourly Demand', 'Northen Region Hourly Demand',
                'Western Region Hourly Demand', 'Eastern Region Hourly Demand',
                'Southern Region Hourly Demand', 'North-Eastern Region Hourly Demand'],
    var_name='location',
    value_name='demand_mw'
)

load_long['location'] = load_long['location'].str.replace(' Hourly Demand', '', regex=False)
load_long['demand_gw'] = load_long['demand_mw'] / 1000
load_long = load_long.drop(columns=['demand_mw'])

load_long.to_parquet('/Users/shreya/Documents/College/Sem 5/ML/Lab/GridSense/data/processed/load_long.parquet', index=False)
load_long.head()

,datetime,location,demand_gw
0,2019-01-01 00:00:00,National,118.69067
1,2019-01-01 01:00:00,National,116.02923
2,2019-01-01 02:00:00,National,114.04414
3,2019-01-01 03:00:00,National,113.64897
4,2019-01-01 04:00:00,National,116.29005
